# Réplicas de MD (parte 2 de 2) — MM/GBSA de las réplicas
Aplica **exactamente** el protocolo de `Colab_MMPBSA_Pareja2.ipynb` (igb=8, 101 snapshots de 500–1000 ps, mismos grupos) a las réplicas r2 y r3 de 2A y 2B generadas en el notebook 1.

1. Celda de conda (reinicia el kernel) → 2. instala GROMACS + gmx_MMPBSA → 3. sube `replicas_md.zip` → 4. corre → 5. descarga `resultados_replicas_mmgbsa.zip`.
Un runtime con CPU basta. Después copia los resultados a `D_F_energia/replicas/resultados/` y corre `python scripts/analisis_replicas.py`.


## 0. Entorno (GROMACS + gmx_MMPBSA)


In [ ]:
# 1. Conda en Colab (reinicia el kernel automáticamente; luego sigue con la celda 2)
!pip install -q "condacolab<0.2"
import condacolab
condacolab.install()


In [ ]:
# 2. GROMACS (CPU, suficiente para gmx_MMPBSA) y entorno separado para gmx_MMPBSA (exige Python <3.13)
!mamba install -y -c conda-forge -c bioconda --no-pin gromacs
!ln -sf /usr/local/bin/gmx /usr/bin/gmx
!conda create -y -q -n gmxMMPBSA -c conda-forge python=3.12 "ambertools>=24.8,<27" "mpi4py>=4.0.1,<5" "numpy>=1.26.4,<2" "matplotlib>=3.8,<4" "scipy>=1.14.1,<2" "pandas>=2.2,<3" "seaborn>=0.13,<0.14" pip
!/usr/local/envs/gmxMMPBSA/bin/python -m pip install -q gmx_MMPBSA==1.7.1
!gmx --version | grep -E "GROMACS version"
!PATH=/usr/local/envs/gmxMMPBSA/bin:$PATH AMBERHOME=/usr/local/envs/gmxMMPBSA gmx_MMPBSA --version


## 1. Subir `replicas_md.zip` y preparar topología seca + `ref.pdb`


In [ ]:
# 3. Subir replicas_md.zip (generado por el notebook 1 de réplicas)
import os, shutil
from google.colab import files

WORK = "/content/work"
shutil.rmtree(WORK, ignore_errors=True)
os.makedirs(WORK, exist_ok=True)

uploaded = files.upload()
zip_name = next(iter(uploaded))
!unzip -q -o "{zip_name}" -d {WORK}
!ls {WORK}


In [ ]:
%%bash
# 4. Crear topol_dry.top (sin SOL ni CL para que coincida con proteina.tpr/proteina.xtc)
#    y generar ref.pdb con cadenas explícitas: A = RNasa A (Protein_chain1), B = Nanobody (Protein_chain2)
set -euo pipefail
WORK=/content/work

for SIS in 2A_r2 2A_r3 2B_r2 2B_r3; do
  cd "$WORK/$SIS"
  grep -vE '^(SOL|CL|NA)[[:space:]]+' topol.top > topol_dry.top
  echo "0" | gmx trjconv -s proteina.tpr -f proteina.xtc -dump 0 -o ref_raw.pdb >/dev/null 2>&1

  python3 - <<'PY'
n_chain1 = 0
in_c1 = False
for line in open("proteina.ndx"):
    s = line.strip()
    if s.startswith("["):
        in_c1 = (s == "[ Protein_chain1 ]")
    elif in_c1 and s:
        n_chain1 += len(s.split())

out = []
atom_idx = 0
for line in open("ref_raw.pdb"):
    if line.startswith(("ATOM  ", "HETATM")):
        atom_idx += 1
        chain = "A" if atom_idx <= n_chain1 else "B"
        line = line[:21] + chain + line[22:]
    out.append(line)
open("ref.pdb", "w").write("".join(out))
print(f"ref.pdb listo: {atom_idx} átomos (Cadena A={n_chain1}, Cadena B={atom_idx - n_chain1})")
PY
done


## 2. D–F. MM/GBSA y descomposición por residuo (`igb=8`, 101 snapshots)

Mismo protocolo para **2A** y **2B**:
- **Receptor:** `Protein_chain2` (Nanobody, cadena B) · **Ligando:** `Protein_chain1` (RNasa A, cadena A)
- **Ventana:** últimos 500 ps (`startframe=501`, `endframe=1001`, `interval=5` → 101 snapshots)
- **Modelo GB:** GB-Neck2 (`igb=8`, radios `mbondi3` / `PBRadii=4`), `saltcon=0.150` M, `temperature=298.15` K
- **Descomposición:** `idecomp=2, dec_verbose=3, csv_format=1, print_res="all"` (todos los residuos para comparar mutados, vecinos, interfaz y antígeno sin perder las mutaciones indirectas).


In [ ]:
%%bash
# 5. Escribir archivo de entrada y ejecutar gmx_MMPBSA para 2A y 2B en paralelo
set -euo pipefail
WORK=/content/work
ENVPY=/usr/local/envs/gmxMMPBSA
export PATH=$ENVPY/bin:$PATH
export AMBERHOME=$ENVPY

run_mmpbsa() {
  SIS=$1
  cd "$WORK/$SIS"
  mkdir -p mmgbsa
  cd mmgbsa
  cp ../topol_dry.top ../topol_Protein_chain_*.itp ../posre_Protein_chain_*.itp .

  cat <<EOT > mmgbsa.in
&general
  sys_name="$SIS", startframe=501, endframe=1001, interval=5,
  temperature=298.15, PBRadii=4, keep_files=0, gmx_path="/usr/local/bin",
/
&gb
  igb=8, saltcon=0.150,
/
&decomp
  idecomp=2, dec_verbose=3, csv_format=1, print_res="all",
/
EOT

  gmx_MMPBSA -O -i mmgbsa.in -cs ../proteina.tpr -ci ../proteina.ndx     -cg Protein_chain2 Protein_chain1 -ct ../proteina.xtc     -cp topol_dry.top -cr ../ref.pdb     -o FINAL_RESULTS_MMPBSA.dat -eo FINAL_RESULTS_MMPBSA.csv     -do FINAL_DECOMP_MMPBSA.dat -deo FINAL_DECOMP_MMPBSA.csv     -nogui > gmx_MMPBSA.log 2>&1 || { tail -40 gmx_MMPBSA.log; exit 1; }
  echo "$SIS: gmx_MMPBSA terminado"
}

# de a dos en paralelo (los runtimes de Colab tienen pocos núcleos)
e=0
run_mmpbsa 2A_r2 & p1=$!; run_mmpbsa 2B_r2 & p2=$!
wait $p1 || e=1; wait $p2 || e=1
run_mmpbsa 2A_r3 & p1=$!; run_mmpbsa 2B_r3 & p2=$!
wait $p1 || e=1; wait $p2 || e=1
exit $e


## Empaquetar resultados

In [ ]:
%%bash
cd /content/work
rm -f resultados_replicas_mmgbsa.zip
zip -qr resultados_replicas_mmgbsa.zip */mmgbsa/FINAL_RESULTS_MMPBSA.* */mmgbsa/FINAL_DECOMP_MMPBSA.* */mmgbsa/mmgbsa.in
ls -lh resultados_replicas_mmgbsa.zip

In [ ]:
from google.colab import files
files.download("/content/work/resultados_replicas_mmgbsa.zip")